# Clase 2: SQL y Pandas (SQLite)

En esta notebook veremos cómo interactuar con bases de datos relacionales utilizando **SQLite** (un motor de base de datos ligero que no requiere instalación de servidor) y cómo integrar estas consultas con **Pandas**.

## 1. Librerías Necesarias

Python ya trae incluida la librería `sqlite3` para trabajar con archivos `.db`.

In [ ]:
import sqlite3
import pandas as pd

## 2. Crear una Base de Datos y una Tabla

Primero creamos una conexión a un archivo de base de datos (si no existe, se creará).

In [ ]:
conn = sqlite3.connect('clase2_ejemplo.db')
cursor = conn.cursor()

# Crear una tabla
cursor.execute('''
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    precio REAL,
    categoria TEXT
)
''')

# Insertar algunos datos
datos = [
    (1, 'Notebook', 1500, 'Electrónica'),
    (2, 'Monitor', 300, 'Electrónica'),
    (3, 'Teclado', 50, 'Electrónica'),
    (4, 'Silla Ergonómica', 200, 'Muebles')
]
cursor.executemany('INSERT OR REPLACE INTO productos VALUES (?,?,?,?)', datos)
conn.commit()
print("Datos insertados correctamente.")

## 3. Leer SQL con Pandas

Pandas permite ejecutar una consulta SQL y obtener los resultados directamente en un DataFrame usando `pd.read_sql`.

In [ ]:
query = "SELECT * FROM productos WHERE precio > 100"
df_filtrado = pd.read_sql(query, conn)
df_filtrado

## 4. Escribir un DataFrame a SQL

También podemos hacer el proceso inverso: llevar un DataFrame a una tabla SQL usando `to_sql`.

In [ ]:
df_ventas = pd.DataFrame({
    'id_producto': [1, 2, 1, 4],
    'cantidad': [10, 5, 2, 8],
    'fecha': ['2026-02-01', '2026-02-02', '2026-02-03', '2026-02-03']
})

df_ventas.to_sql('ventas', conn, if_exists='replace', index=False)
print("DataFrame guardado en la base de datos como la tabla 'ventas'.")

## 5. Consultas Combinadas (Joins)

Podemos usar toda la potencia de SQL directamente desde Pandas.

In [ ]:
query_final = """
SELECT v.fecha, p.nombre, p.precio, v.cantidad, (p.precio * v.cantidad) as total
FROM ventas v
JOIN productos p ON v.id_producto = p.id
"""
df_reporte = pd.read_sql(query_final, conn)
df_reporte

## 6. Explorando la Estructura (Metadata)

A veces recibimos una base de datos sin saber qué tablas contiene. En SQLite, podemos consultar una tabla especial llamada `sqlite_master` para obtener esta información.

Además, para no saturar la memoria, usamos `LIMIT` para traer solo unas pocas filas de prueba.

In [ ]:
# 1. Listar todas las tablas de la base de datos
query_tablas = "SELECT name FROM sqlite_master WHERE type='table';"
tablas = pd.read_sql(query_tablas, conn)
print("Tablas encontradas:")
print(tablas)

# 2. Ver las primeras 3 filas de una tabla específica
query_preview = "SELECT * FROM productos LIMIT 3"
df_preview = pd.read_sql(query_preview, conn)
print("\nVista previa de 'productos':")
print(df_preview)

# 3. Usar Pandas para entender los tipos de datos de la tabla
print("\nInformación de la tabla:")
df_preview.info()

## Trabajo en Clase: Exploración de una Base de Datos Desconocida

**Desafío**: Imagina que recibes un archivo de base de datos de un colega pero no tienes la documentación de qué tablas contiene ni qué datos hay en ellas. Tu objetivo es explorar la estructura y extraer información útil.

**Instrucciones**:
1. **Preparación**: Descargar la base de datos `ecommerce_desconocido.db` de [este link](https://drive.google.com/file/d/1XluiJsmmxjnphUfU1VF46IV-tqrk9ZU8/view?usp=sharing) y subirla al entorno de Colab.
2. **Conexión**: Conectarse a la base de datos creada.
   - *Tip: `sqlite3.connect('nombre.db')`.*
3. **Descubrimiento**: Listar los nombres de todas las tablas existentes.
   - *Tip: Consulta la tabla `sqlite_master`.*
4. **Inspección**: Elegir una tabla y usar `pd.read_sql` con un `LIMIT 5` para entender las columnas.
5. **Análisis**: Cargar la tabla completa en un DataFrame y utilizar el método `.info()` o `.describe()`.
6. **Extracción**: Realizar una consulta SQL que filtre los datos por alguna condición.
   - *Tip: Prueba un `SELECT * FROM tabla WHERE condicion`.*
7. **Avanzado**: Construir un dataframe a partir de un join complejo, por ejemplo, cantidad de productos y nombre de cliente por pedido.
   - *Tip: La query de `read_sql` puede contener joins y cualquier sql compatible con SQLite. Se recomienda probar consultas avanzadas.*

**Resultado esperado**: El código debe mostrar la lista de tablas encontradas y un breve resumen estadístico de la tabla principal analizada.

In [ ]:
# Escribir el código a partir de acá


In [ ]:
# Cerramos la conexión al terminar
conn.close()